In [3]:
import numpy as np
import pickle

from pathlib import Path

from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

In [4]:
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../saved_models")

In [5]:
with open(MODEL_DIR / "tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

print("Vocabulary Size:", len(tokenizer.word_index))

Vocabulary Size: 77160


In [6]:
model = load_model(MODEL_DIR / "chatbot_model.keras")

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 20, 128)   │  9,876,608 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 20)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 20, 128)   │  9,876,608 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 20, 256), │    394,240 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 20, 77161) │ 19,830,377 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 121,116,221 (462.02 MB)

 Trainable params: 40,372,073 (154.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 80,744,148 (308.01 MB)

In [7]:
encoder_inputs = model.input[0]

encoder_embedding = model.get_layer("encoder_embedding")
encoder_lstm = model.get_layer("encoder_lstm")

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding(encoder_inputs)
)

encoder_model = Model(
    encoder_inputs,
    [state_h, state_c]
)

In [8]:
decoder_inputs = model.input[1]

decoder_embedding = model.get_layer("decoder_embedding")
decoder_lstm = model.get_layer("decoder_lstm")
decoder_dense = model.get_layer("decoder_output")

In [9]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_output = decoder_embedding(decoder_inputs)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding_output,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]

decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [10]:
reverse_word_index = {
    index: word
    for word, index in tokenizer.word_index.items()
}

print("Vocabulary:", len(reverse_word_index))

Vocabulary: 77160


In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LENGTH = 20

def encode_input(sentence):
    sentence = sentence.lower()

    sequence = tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post"
    )

    states = encoder_model.predict(sequence, verbose=0)

    return states

In [12]:
def generate_response(sentence):
    states = encode_input(sentence)

    target_seq = np.zeros((1, 1))

    target_seq[0, 0] = tokenizer.word_index["sos"]

    decoded_sentence = ""

    while True:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_word_index.get(sampled_token_index, "")

        if sampled_word == "eos" or len(decoded_sentence.split()) > MAX_LENGTH:
            break

        decoded_sentence += sampled_word + " "

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        states = [h, c]

    return decoded_sentence.strip()

In [13]:
print(generate_response("hello"))

tyler gordon. did not you? <EOS>  have to go home. <EOS>  pull the plug. <EOS>  pull it out. <EOS>  look <EOS>


In [14]:
print(generate_response("how are you"))

s a mistake. i am not going to tell you that i am not a cop anymore. i have been working


In [15]:
print(generate_response("what is your name"))

s a lot of a secret compartment, perhaps containing the people have up, heh? roll one <EOS>  all night. <EOS>  look


In [16]:
print(generate_response("good morning"))

tom, i am a little excited, a lot of things. i am sorry for you. <EOS>  look at you. <EOS>  pull
